In [6]:
from tasks import cmj

cmj_data = ["keypoints/cmj/P03_CMJBL_FRONT.txt"]   # one path per capture

all_jumps_df_ptmh = cmj.get_jump_heights(cmj_data, ptm='height')
all_jumps_df_ptmg = cmj.get_jump_heights(cmj_data, ptm='gravity')

FileNotFoundError: [Errno 2] No such file or directory: 'keypoints/cmj/P03_CMJBL_FRONT.txt'

In [7]:
from tasks import cmj
print(cmj.__file__)
print([n for n in dir(cmj) if not n.startswith("_")])

c:\Users\dylan\Documents\Projects\machine_vision_media_pipe\tasks\cmj.py
['COORDS', 'METRE_HEIGHTS', 'PER_FRAME', 'POSE_LANDMARKS', 'PTM', 'SUBJECT_HEIGHTS', 'calculate_cmj', 'flip_axis', 'get_jump_heights', 'get_reps', 'height_scale', 'load_pose_file', 'np', 'os', 'pd', 'pixel_body_height', 're', 'savgol_filter', 'subject_from_filename', 'to_pixels', 'ts_jump_height']


In [3]:
print(all_jumps_df_ptmg)
print(all_jumps_df_ptmh)

             ptm  reps                heights  mean   best
subject                                                   
P03      gravity     3  [25.37, 25.84, 26.79]  26.0  26.79
            ptm  reps                heights   mean   best
subject                                                   
P03      height     3  [24.96, 25.25, 25.02]  25.08  25.25


In [ ]:
import glob, os
import numpy as np
from tasks import cmj

# 1. gather the files — print and eyeball before running
cmj_data = sorted(glob.glob("keypoints/cmj/*CMJBL*.txt"))
print(len(cmj_data), "files")
for p in cmj_data:
    print("  ", os.path.basename(p))

# 2. run both scaling methods over the whole cohort in one call each
#    (gravity clamp needs all subjects together, so don't loop per-file)
ptmh = cmj.get_jump_heights(cmj_data, ptm='height')
ptmg = cmj.get_jump_heights(cmj_data, ptm='gravity')

# 3. health check — catch the two things that break silently
print("\n-- rep counts (want 3 each) --")
print(ptmh[ptmh["reps"] != 3][["reps"]] if (ptmh["reps"] != 3).any() else "all 3 ✓")

bad_grav = ptmg[ptmg["heights"].apply(lambda h: any(np.isnan(x) for x in h))]
print("\n-- subjects with a failed gravity fit --")
print(bad_grav.index.tolist() if len(bad_grav) else "none ✓")

# 4. save so you don't recompute on every kernel restart
os.makedirs("results", exist_ok=True)
ptmh.to_csv("results/cmj_ptmh.csv")
ptmg.to_csv("results/cmj_ptmg.csv")

ptmh

In [4]:
import glob, os, re

all_txt = glob.glob("keypoints/cmj/*.txt")

def is_bilateral_camera(path):
    name = os.path.basename(path).upper()
    if "WORLD" in name:                 # drop the world-coordinate twins
        return False
    # bilateral = has 'BL', not 'UL'.  matches CMJBL, CMJ_BL, CMJ BL, etc.
    return re.search(r"\bUL\b|UL_|_UL|CMJUL", name) is None and \
           re.search(r"BL", name) is not None

cmj_data = sorted(p for p in all_txt if is_bilateral_camera(p))

print(len(cmj_data), "files")
for p in cmj_data:
    print("  ", os.path.basename(p))

16 files
   P03_CMJBL_FRONT.txt
   P04_CMJ_BL_FRONT.txt
   P05_CMJ_BL_FRONT.txt
   P06_CMJ_BL_FRONT.txt
   P07_CMJ_BL_FRONT.txt
   P08_CMJ_BL_FRONT.txt
   P09_CMJ_BL_FRONT.txt
   P10_CMJ_BL_FRONT.txt
   P11_CMJ_BL_FRONT.txt
   P12_CMJ_BL_FRONT.txt
   P13_CMJ_BL_FRONT.txt
   P14_CMJ_BL_FRONT.txt
   P15_CMJ_BL_FRONT.txt
   P16_CMJ_BL_FRONT.txt
   P17_CMJ_BL_FRONT.txt
   P18_CMJ_BL_FRONT.txt


In [6]:
import numpy as np
ptmh = cmj.get_jump_heights(cmj_data, ptm='height')
ptmg = cmj.get_jump_heights(cmj_data, ptm='gravity')

# 3. health check — catch the two things that break silently
print("\n-- rep counts (want 3 each) --")
print(ptmh[ptmh["reps"] != 3][["reps"]] if (ptmh["reps"] != 3).any() else "all 3 ✓")

bad_grav = ptmg[ptmg["heights"].apply(lambda h: any(np.isnan(x) for x in h))]
print("\n-- subjects with a failed gravity fit --")
print(bad_grav.index.tolist() if len(bad_grav) else "none ✓")

# 4. save so you don't recompute on every kernel restart
os.makedirs("results", exist_ok=True)
ptmh.to_csv("results/cmj_ptmh.csv")
ptmg.to_csv("results/cmj_ptmg.csv")


-- rep counts (want 3 each) --
all 3 ✓

-- subjects with a failed gravity fit --
none ✓


In [7]:
cmj.ba_plots(all_jumps_df_ptmh, title='height PTM')
cmj.ba_plots(all_jumps_df_ptmg, title='gravity PTM')
#dj.ba_plots(dj_data, ptm='height')
#dj.ba_plots(dj_data, ptm='gravity')
results_df_g = cmj.get_metrics(all_jumps_df_ptmg, ptm='gravity')
results_df_h = cmj.get_metrics(all_jumps_df_ptmh, ptm='height')
#dj_metrics_h = dj.get_metrics(dj_data, ptm='height')


AttributeError: module 'tasks.cmj' has no attribute 'ba_plots'

In [2]:
from utilities import utils
cmj_data = utils.read_list('cmj_data')
print(type(cmj_data), len(cmj_data))
print(cmj_data[0].keys())
print(cmj_data[0]['bl'].keys())
print(cmj_data[0]['bl']['hip'].keys())     # 'coda' and 'op' present?
# and hunt for the force field:
print([k for k in cmj_data[0].keys()])
print([k for k in cmj_data[0]['bl'].keys()])

<class 'list'> 16
dict_keys(['ul', 'bl'])
dict_keys(['force', 'hip', 'knee', 'ankle', 'toe'])
dict_keys(['coda', 'op'])
['ul', 'bl']
['force', 'hip', 'knee', 'ankle', 'toe']


In [3]:
f = cmj_data[0]['bl']['force']
print(type(f), np.shape(f) if hasattr(f, '__len__') else f)
print(f[:50] if hasattr(f, '__len__') else f)

<class 'dict'> ()


KeyError: slice(None, 50, None)

In [4]:
import numpy as np
f = cmj_data[0]['bl']['force']
print("type:", type(f))
print("shape/len:", np.shape(f) if hasattr(f, '__len__') else "scalar")
print("first values:", np.array(f).ravel()[:20])

# also confirm what the OpenPose path did with it:
import inspect
print(inspect.getsource(utils.force_flight_time))
print(inspect.getsource(utils.jump_heights))

type: <class 'dict'>
shape/len: ()
first values: [{1: array([ 0.4302026 ,  0.40541679,  0.37646982, ..., -0.20814802,
        -0.06833682,  0.08505051], shape=(23103,)), 2: array([ -0.99745536,  -0.26254941,   0.23908414, ..., 236.28188805,
        237.14798766, 238.33127042], shape=(23103,)), 3: array([ 0.70460037,  1.34911735,  2.03853101, ..., -1.38448196,
        -0.14969889,  1.04977755], shape=(23103,)), 4: array([   3.66510499,    3.48827897,    3.38205945, ..., -228.52759503,
        -225.03152791, -221.2794587 ], shape=(23103,))}                                                                       ]
def force_flight_time(subjects, thresh=40, plot=False):
    force_ft = {'ul': [], 'bl': []}
    for s, subject in enumerate(subjects):
        if plot:
            fig, axs = plt.subplots(2, 2, figsize=(10, 5),
                                    dpi=100, sharex='col')
        for i, task in enumerate(['bl', 'ul']):
            F = subject[task]['force']
            f1 = F[1] + F[

In [9]:
from tasks import cmj
import inspect
print(inspect.getsource(utils.force_flight_time))   # the full thing, not truncated
print(inspect.getsource(cmj.get_metrics))
print(inspect.getsource(cmj.ba_plots))

def force_flight_time(subjects, thresh=40, plot=False):
    force_ft = {'ul': [], 'bl': []}
    for s, subject in enumerate(subjects):
        if plot:
            fig, axs = plt.subplots(2, 2, figsize=(10, 5),
                                    dpi=100, sharex='col')
        for i, task in enumerate(['bl', 'ul']):
            F = subject[task]['force']
            f1 = F[1] + F[3]
            f2 = F[2] + F[4]
            f = f1 if f1.max() > f2.max() else f2
            force = f - f.min()
            if thresh == 'auto':
                thresh = force[:1000].mean()/10
            toe = F = subject[task]['toe']['coda'][:,2]
            toe = resample(toe, len(f))
            bases = np.where(force<=thresh)[0]
            force_diffs = np.diff(bases, prepend=0)
            second_ = np.where(force_diffs>100)[0]
            second_diffs = np.diff(second_)
            last_jump_time = len(force_diffs) - second_diffs.sum()
            T = np.hstack([second_diffs, last_jump_time])
       

AttributeError: module 'tasks.cmj' has no attribute 'get_metrics'